In [1]:
import pandas as pd
import numpy as np
import os

# load the saved data - notice we never call yfinance again
raw = pd.read_parquet("../data/sp500_raw.parquet")

print(f"loaded data: {raw.shape}")
print(f"date range: {raw.index.min()} to {raw.index.max()}")

loaded data: (1509, 2518)
date range: 2019-01-02 00:00:00 to 2024-12-30 00:00:00


In [2]:
# we only need closing prices for feature engineering
close = raw["Close"].copy()

# drop columns that are entirely NaN (the 3 delisted stocks)
close = close.dropna(axis=1, how="all")

print(f"close prices shape: {close.shape}")
print(f"stocks available: {close.shape[1]}")
print(close.head(3))

close prices shape: (1509, 500)
stocks available: 500
Ticker              A       AAPL       ABBV  ABNB        ABT       ACGL  \
Date                                                                      
2019-01-02  62.300152  37.469208  64.789429   NaN  60.801193  24.904034   
2019-01-03  60.005020  33.736996  62.654716   NaN  57.931736  24.514164   
2019-01-04  62.082005  35.177204  64.673256   NaN  59.585197  25.094212   

Ticker             ACN        ADBE        ADI        ADM  ...         WY  \
Date                                                      ...              
2019-01-02  126.118225  224.570007  75.146545  32.823856  ...  16.126350   
2019-01-03  121.812378  215.699997  70.607323  32.679558  ...  16.673128   
2019-01-04  126.548866  226.190002  72.321518  33.433033  ...  17.242380   

Ticker           WYNN        XEL        XOM        XYL        XYZ        YUM  \
Date                                                                           
2019-01-02  96.975555  38.741

In [3]:
# ── FEATURE ENGINEERING ──────────────────────────────────────────
# For each stock, on each day, we compute signals that might
# predict whether it will outperform next month

features = pd.DataFrame(index=close.index)

# 1. MOMENTUM SIGNALS
# "has this stock been going up recently?"

# 12-1 month momentum (skip last month to avoid reversal noise)
# 252 trading days in a year, 21 in a month
mom_12_1 = close.pct_change(252) - close.pct_change(21)

# 6-1 month momentum
mom_6_1 = close.pct_change(126) - close.pct_change(21)

# 1 month return (short term reversal - stocks that shot up tend to pull back)
mom_1m = close.pct_change(21)

# 2. VOLATILITY SIGNALS
# "how wildly has this stock been moving?"

# 1 month realised volatility
vol_1m = close.pct_change().rolling(21).std()

# 3 month realised volatility  
vol_3m = close.pct_change().rolling(63).std()

print("signals computed")
print(f"mom_12_1 shape: {mom_12_1.shape}")
print(mom_12_1.tail(3))

signals computed
mom_12_1 shape: (1509, 500)
Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2024-12-26 -0.031843  0.236004  0.189250  0.015297  0.100566  0.400717   
2024-12-27 -0.032745  0.242415  0.208635  0.021821  0.093442  0.400851   
2024-12-30 -0.004827  0.235692  0.217812  0.008906  0.093831  0.392669   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2024-12-26  0.038580 -0.115363  0.135655 -0.230368  ... -0.041584  0.036499   
2024-12-27  0.042127 -0.100925  0.112162 -0.215414  ... -0.053195  0.023839   
2024-12-30  0.045377 -0.119266  0.096617 -0.203094  ... -0.049049  0.009345   

Ticker           XEL       XOM       XYL       XYZ       YUM       ZBH  \
Date                                                                     
2024-12-26  0.185261  0.190514  0.132821

In [4]:
def cross_sectional_rank(df):
    """Rank each stock against all others on each day"""
    return df.rank(axis=1, pct=True)

# rank all signals
mom_12_1_rank = cross_sectional_rank(mom_12_1)
mom_6_1_rank  = cross_sectional_rank(mom_6_1)
mom_1m_rank   = cross_sectional_rank(mom_1m)
vol_1m_rank   = cross_sectional_rank(vol_1m)
vol_3m_rank   = cross_sectional_rank(vol_3m)

# stack everything into one feature dataframe
# rows = (date, ticker), columns = features
feature_dict = {
    "mom_12_1": mom_12_1_rank,
    "mom_6_1":  mom_6_1_rank,
    "mom_1m":   mom_1m_rank,
    "vol_1m":   vol_1m_rank,
    "vol_3m":   vol_3m_rank,
}

# combine into a long-format dataframe
feat_df = pd.concat(
    {k: v.stack() for k, v in feature_dict.items()},
    axis=1
)
feat_df.index.names = ["Date", "Ticker"]

print(f"feature dataframe shape: {feat_df.shape}")
print(feat_df.tail(5))

feature dataframe shape: (732329, 5)
                   mom_12_1  mom_6_1  mom_1m  vol_1m  vol_3m
Date       Ticker                                           
2024-09-23 GEV          NaN      NaN   0.996   0.892   0.926
           SOLV         NaN      NaN   0.970   0.740   0.766
2024-09-24 GEV          NaN      NaN   0.998   0.886   0.926
           SOLV         NaN      NaN   0.970   0.732   0.768
2024-09-25 GEV          NaN      NaN   0.996   0.882   0.928


In [5]:
# ── LABEL CONSTRUCTION ───────────────────────────────────────────
# What are we predicting? Which stocks will outperform
# their peers over the NEXT 21 trading days (1 month)

# 1. compute raw 1-month forward return for every stock
forward_return = close.pct_change(21).shift(-21)

# 2. subtract the cross-sectional mean each day
# so we're predicting relative performance, not market direction
market_return = forward_return.mean(axis=1)
excess_return = forward_return.sub(market_return, axis=0)

# 3. winsorise - clip extreme values at 5th and 95th percentile
# prevents a few crazy outliers from dominating the model
def winsorise(df, lower=0.05, upper=0.95):
    low  = df.quantile(lower, axis=1)
    high = df.quantile(upper, axis=1)
    return df.clip(lower=low, upper=high, axis=0)

excess_return_w = winsorise(excess_return)

# 4. stack into long format to match feat_df
labels = excess_return_w.stack()
labels.index.names = ["Date", "Ticker"]
labels.name = "label"

print(f"labels shape: {labels.shape}")
print(labels.describe())

labels shape: (732329,)
count    732329.000000
mean         -0.001157
std           0.067166
min          -0.285669
25%          -0.045527
50%          -0.002655
75%           0.041490
max           0.375833
Name: label, dtype: float64


In [6]:
# ── COMBINE FEATURES + LABELS ─────────────────────────────────────
# Join on (Date, Ticker) index

ml_data = feat_df.join(labels, how="inner")

# drop rows where ANY feature or label is NaN
ml_data = ml_data.dropna()

print(f"final dataset shape: {ml_data.shape}")
print(f"date range: {ml_data.index.get_level_values('Date').min()} "
      f"to {ml_data.index.get_level_values('Date').max()}")
print(f"avg stocks per day: {len(ml_data) / ml_data.index.get_level_values('Date').nunique():.0f}")
print(ml_data.tail(5))

final dataset shape: (606490, 6)
date range: 2020-01-02 00:00:00 to 2024-11-27 00:00:00
avg stocks per day: 491
                   mom_12_1  mom_6_1  mom_1m  vol_1m  vol_3m     label
Date       Ticker                                                     
2024-11-27 XYZ     0.395582    0.578   0.948   0.944   0.904  0.045113
           YUM     0.246988    0.218   0.398   0.110   0.098  0.022700
           ZBH     0.080321    0.108   0.618   0.508   0.572  0.002777
           ZBRA    0.893574    0.762   0.500   0.214   0.482  0.005782
           ZTS     0.156627    0.382   0.172   0.360   0.268 -0.022286


In [7]:
ml_data.to_parquet("../data/ml_data.parquet")
print("saved to data/ml_data.parquet")

saved to data/ml_data.parquet


In [8]:
# ── ADDITIONAL FEATURES ───────────────────────────────────────────

# 1. Price to 52-week high
# how close is the stock to its yearly peak?
# stocks near highs tend to keep going - "52-week high effect"
high_52w = close.rolling(252).max()
price_to_high = close / high_52w

# 2. Short term reversal - 5 day return
# very short term mean reversion signal
reversal_5d = close.pct_change(5)

# 3. Volume trend
# is volume increasing? rising volume confirms price moves
volume = raw["Volume"].copy()
volume = volume.dropna(axis=1, how="all")

# make sure volume columns match close columns
common_tickers = close.columns.intersection(volume.columns)
volume = volume[common_tickers]

vol_trend = volume.rolling(21).mean() / volume.rolling(63).mean()

# 4. Volatility ratio - short vol / long vol
# ratio > 1 means stock is getting more volatile recently
vol_ratio = vol_1m / vol_3m

print("new features computed")
print(f"price_to_high shape : {price_to_high.shape}")
print(f"reversal_5d shape   : {reversal_5d.shape}")
print(f"vol_trend shape     : {vol_trend.shape}")
print(f"vol_ratio shape     : {vol_ratio.shape}")

new features computed
price_to_high shape : (1509, 500)
reversal_5d shape   : (1509, 500)
vol_trend shape     : (1509, 500)
vol_ratio shape     : (1509, 500)


In [9]:
# ── ADDITIONAL FEATURES ───────────────────────────────────────────

# 1. Price to 52-week high
# how close is the stock to its yearly peak?
# stocks near highs tend to keep going - "52-week high effect"
high_52w = close.rolling(252).max()
price_to_high = close / high_52w

# 2. Short term reversal - 5 day return
# very short term mean reversion signal
reversal_5d = close.pct_change(5)

# 3. Volume trend
# is volume increasing? rising volume confirms price moves
volume = raw["Volume"].copy()
volume = volume.dropna(axis=1, how="all")

# make sure volume columns match close columns
common_tickers = close.columns.intersection(volume.columns)
volume = volume[common_tickers]

vol_trend = volume.rolling(21).mean() / volume.rolling(63).mean()

# 4. Volatility ratio - short vol / long vol
# ratio > 1 means stock is getting more volatile recently
vol_ratio = vol_1m / vol_3m

print("new features computed")
print(f"price_to_high shape : {price_to_high.shape}")
print(f"reversal_5d shape   : {reversal_5d.shape}")
print(f"vol_trend shape     : {vol_trend.shape}")
print(f"vol_ratio shape     : {vol_ratio.shape}")

new features computed
price_to_high shape : (1509, 500)
reversal_5d shape   : (1509, 500)
vol_trend shape     : (1509, 500)
vol_ratio shape     : (1509, 500)


In [10]:
# ── RANK NEW FEATURES + REBUILD ML_DATA ──────────────────────────

# rank new features cross-sectionally same as before
price_to_high_rank = cross_sectional_rank(price_to_high)
reversal_5d_rank   = cross_sectional_rank(reversal_5d)
vol_ratio_rank     = cross_sectional_rank(vol_ratio)

# volume trend - only rank over matching tickers
vol_trend_rank = cross_sectional_rank(vol_trend)

# rebuild feature dict with all 9 features
feature_dict_v2 = {
    "mom_12_1"      : mom_12_1_rank,
    "mom_6_1"       : mom_6_1_rank,
    "mom_1m"        : mom_1m_rank,
    "vol_1m"        : vol_1m_rank,
    "vol_3m"        : vol_3m_rank,
    "price_to_high" : price_to_high_rank,
    "reversal_5d"   : reversal_5d_rank,
    "vol_ratio"     : vol_ratio_rank,
}

# stack into long format
feat_df_v2 = pd.concat(
    {k: v.stack() for k, v in feature_dict_v2.items()},
    axis=1
)
feat_df_v2.index.names = ["Date", "Ticker"]

# combine with labels
ml_data_v3 = feat_df_v2.join(labels, how="inner").dropna()

print(f"old feature count : 5")
print(f"new feature count : {len(feature_dict_v2)}")
print(f"old ml_data shape : (606490, 6)")
print(f"new ml_data shape : {ml_data_v3.shape}")
print(ml_data_v3.tail(3))

old feature count : 5
new feature count : 8
old ml_data shape : (606490, 6)
new ml_data shape : (606490, 9)
                   mom_12_1  mom_6_1  mom_1m  vol_1m  vol_3m  price_to_high  \
Date       Ticker                                                             
2024-11-27 ZBH     0.080321    0.108   0.618   0.508   0.572       0.180723   
           ZBRA    0.893574    0.762   0.500   0.214   0.482       0.936747   
           ZTS     0.156627    0.382   0.172   0.360   0.268       0.267068   

                   reversal_5d  vol_ratio     label  
Date       Ticker                                    
2024-11-27 ZBH           0.420      0.346  0.002777  
           ZBRA          0.862      0.082  0.005782  
           ZTS           0.176      0.592 -0.022286  


In [11]:
ml_data_v3.to_parquet("../data/ml_data_v3.parquet")
print("saved ml_data_v3.parquet")

saved ml_data_v3.parquet
